In [ ]:
import librosa
import librosa.display as dsp
from IPython.display import Audio
import pandas as pd
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [ ]:
# 1. Quay về thư mục gốc của Colab để đảm bảo vị trí đúng
%cd /content

# 2. Xóa thư mục dự án cũ đi nếu nó tồn tại (đây chính là bước "ghi đè")
PROJECT_DIR = 'project_dir'
if os.path.exists(PROJECT_DIR):
    print(f"Thư mục '{PROJECT_DIR}' đã tồn tại. Đang xóa để ghi đè...")
    !rm -rf {PROJECT_DIR}

# 3. Tạo lại thư mục dự án và di chuyển vào đó
print(f"Tạo thư mục mới '{PROJECT_DIR}'...")
!mkdir {PROJECT_DIR}
%cd {PROJECT_DIR}

# 4. Khởi tạo git và tải code bằng sparse checkout
print("Đang tải code từ GitHub...")
!git init
!git remote add origin https://github.com/ChiThanh512/Machine-Learning-Project-251---CEML1.git
!git config core.sparsecheckout true
!echo "HPMR/" >> .git/info/sparse-checkout
!git pull origin main

# 5. Kiểm tra kết quả
print("\n--- Cấu trúc thư mục sau khi tải: ---")
!ls -R

In [ ]:
# Thêm đường dẫn đến thư mục HPMR vào sys.path
hpmr_path = os.path.abspath('HPMR')
if hpmr_path not in sys.path:
    sys.path.append(hpmr_path)
    print(f"Đã thêm '{hpmr_path}' vào sys.path")

# Import hàm download từ module data_loader
from modules.data_loader import download_kaggle_dataset
from modules.preprocessing import read_dataset_and_save_feture
!pip install hmmlearn

In [ ]:
# --- Cấu hình ---
KAGGLE_USER = 'nguyenk512'
KAGGLE_API_KEY = '187454a718c857637f7319f39e33b509'
DATASET_TO_DOWNLOAD = 'subhajournal/free-spoken-digit-database'
TARGET_DIRECTORY = './HPMR/spoken_digit_data' # Thư mục sẽ được tạo bên trong project_dir

# --- Gọi hàm đã import ---
download_kaggle_dataset(
    dataset_name=DATASET_TO_DOWNLOAD,
    username=KAGGLE_USER,
    key=KAGGLE_API_KEY,
    download_dir=TARGET_DIRECTORY
)

In [ ]:
# Đường dẫn đến thư mục dữ liệu đã tải về
DATA_FOLDER = './HPMR/spoken_digit_data' 

# Đường dẫn để lưu file .npz kết quả vào thư mục 'features'
SAVE_FILE_PATH_NPZ = './HPMR/features/processed_data.npz'

# Gọi hàm để bắt đầu quá trình
read_dataset_and_save_feture(root_folder_path=DATA_FOLDER, save_path=SAVE_FILE_PATH_NPZ)

# 1 EDA

In [ ]:
## Định nghĩa các tham số cơ bản
SR = 22050 #tần số lấy mẫu
N_FFT = 512 #int(0.025*SR) # khoảng lấy mẫu fft 25ms
N_HOP = 256 #int(0.010*SR) # bước nhảy giữa 2 frame
N_MFCC = 13
N_MELS =40
pre_emphasis = 0.95

In [ ]:
from EDA.py import *
df = create_dataframe_from_folders(DATA_FOLDER)
df.head()

In [ ]:
get_random_audio(df,0)

In [ ]:
get_random_audio(df,1)

In [ ]:
get_random_audio(df,2)

In [ ]:
get_random_audio(df,3)

In [ ]:
get_random_audio(df,4)

In [ ]:
get_random_audio(df,5)

In [ ]:
get_random_audio(df,6)

In [ ]:
get_random_audio(df,7)

In [ ]:
get_random_audio(df,8)

In [ ]:
get_random_audio(df,9)

In [ ]:
from modules.preprocessing import load_and_preprocess_data
from modules.preprocessing import split_train_test
 
# Đường dẫn đến file dữ liệu
DATA_FILE_PATH_NPZ = './HPMR/features/processed_data.npz'

# 1. Tải và tiền xử lý
X_processed, y_processed, class_names = load_and_preprocess_data(DATA_FILE_PATH_NPZ)

# 2. Chia dữ liệu
X_train, X_test, y_train, y_test = split_train_test(X_processed, y_processed)

In [ ]:
from modules.training import train_and_evaluate_continue_hmm, save_hmm_models
import os

# 3. Huấn luyện và đánh giá mô hình
models, y_pred, metrics = train_and_evaluate_continue_hmm(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    class_names=class_names,
    num_states=5,
    n_loop=30,
    tol=1e-3
)

# 4. Lưu mô hình vào thư mục ./HPMR/models
model_save_path = './HPMR/models/continue_hmm.pkl'
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
save_hmm_models(models, class_names, filepath=model_save_path)

print(f"\n✅ Model đã được lưu tại: {model_save_path}")
print(f"📊 Tổng kết:")
print(f"   - Accuracy: {metrics['accuracy']:.4f}")
print(f"   - Precision (Macro): {metrics['precision_macro']:.4f}")
print(f"   - Recall (Macro): {metrics['recall_macro']:.4f}")
print(f"   - F1-Score (Macro): {metrics['f1_macro']:.4f}")